In [1]:
#Import all the packages required for the program to run
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.cluster import KMeans
from sklearn import metrics

import matplotlib.pyplot as plt
from matplotlib import pyplot
import seaborn as sn

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import multiprocessing 
from sklearn.model_selection import GridSearchCV
from sklearn import svm
import time

In [2]:
#change working directory
import os
os.getcwd()
#os.chdir("Flight_Ticket_Participant_Datasets")

#load the dataset
flight_train_df = pd.read_excel("Data_Train.xlsx", na_values= '?')

In [3]:
flight_train_df.shape

(10683, 11)

In [4]:
flight_train_df['day_of_week'] = pd.to_datetime(flight_train_df['Date_of_Journey']).dt.day_name()
days = {'Monday':1, 'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7}
flight_train_df['day_of_week'] = flight_train_df['day_of_week'].apply(lambda x: days[x])
flight_train_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day_of_week
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,7
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,6
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,5
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,4
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,4


In [5]:
flight_train_df.drop(columns='Route', inplace=True)

In [6]:
flight_train_df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        1
Additional_Info    0
Price              0
day_of_week        0
dtype: int64

In [7]:
null_columns=flight_train_df.columns[flight_train_df.isnull().any()]
flight_train_df[null_columns].isnull().sum()

Total_Stops    1
dtype: int64

In [8]:
print(flight_train_df[flight_train_df["Total_Stops"].isnull()][null_columns])

     Total_Stops
9039         NaN


In [9]:
flight_train_df.iloc[9039,0:10]

Airline               Air India
Date_of_Journey       6/05/2019
Source                    Delhi
Destination              Cochin
Dep_Time                  09:45
Arrival_Time       09:25 07 May
Duration                23h 40m
Total_Stops                 NaN
Additional_Info         No info
Price                      7480
Name: 9039, dtype: object

In [10]:
flight_train_df.drop(index=9039, inplace=True)

In [11]:
flight_train_df.shape

(10682, 11)

In [12]:
flight_train_df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        0
Additional_Info    0
Price              0
day_of_week        0
dtype: int64

In [13]:
flight_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10682 entries, 0 to 10682
Data columns (total 11 columns):
Airline            10682 non-null object
Date_of_Journey    10682 non-null object
Source             10682 non-null object
Destination        10682 non-null object
Dep_Time           10682 non-null object
Arrival_Time       10682 non-null object
Duration           10682 non-null object
Total_Stops        10682 non-null object
Additional_Info    10682 non-null object
Price              10682 non-null int64
day_of_week        10682 non-null int64
dtypes: int64(2), object(9)
memory usage: 1001.4+ KB


In [14]:
flight_train_df['Airline'].unique()

array(['IndiGo', 'Air India', 'Jet Airways', 'SpiceJet',
       'Multiple carriers', 'GoAir', 'Vistara', 'Air Asia',
       'Vistara Premium economy', 'Jet Airways Business',
       'Multiple carriers Premium economy', 'Trujet'], dtype=object)

In [15]:
le_Airline = preprocessing.LabelEncoder()
le_Airline.fit(flight_train_df['Airline'])
flight_train_df['Airline'] = le_Airline.transform(flight_train_df['Airline'])
le_name_mapping = dict(zip(le_Airline.classes_, le_Airline.transform(le_Airline.classes_)))
print(le_name_mapping)
#flight_train_df = flight_train_df.append(flight_train_df)

{'Air Asia': 0, 'Air India': 1, 'GoAir': 2, 'IndiGo': 3, 'Jet Airways': 4, 'Jet Airways Business': 5, 'Multiple carriers': 6, 'Multiple carriers Premium economy': 7, 'SpiceJet': 8, 'Trujet': 9, 'Vistara': 10, 'Vistara Premium economy': 11}


In [16]:
flight_train_df['Source'].unique()

array(['Banglore', 'Kolkata', 'Delhi', 'Chennai', 'Mumbai'], dtype=object)

In [17]:
le_Source = preprocessing.LabelEncoder()
le_Source.fit(flight_train_df['Source'])
flight_train_df['Source'] = le_Source.transform(flight_train_df['Source'])
le_name_mapping = dict(zip(le_Source.classes_, le_Source.transform(le_Source.classes_)))
print(le_name_mapping)
#flight_train_df = flight_train_df.append(flight_train_df)

{'Banglore': 0, 'Chennai': 1, 'Delhi': 2, 'Kolkata': 3, 'Mumbai': 4}


In [18]:
flight_train_df['Destination'].unique()

array(['New Delhi', 'Banglore', 'Cochin', 'Kolkata', 'Delhi', 'Hyderabad'],
      dtype=object)

In [19]:
le_Destination = preprocessing.LabelEncoder()
le_Destination.fit(flight_train_df['Destination'])
flight_train_df['Destination'] = le_Destination.transform(flight_train_df['Destination'])
le_name_mapping = dict(zip(le_Destination.classes_, le_Destination.transform(le_Destination.classes_)))
print(le_name_mapping)
#flight_train_df = flight_train_df.append(flight_train_df)

{'Banglore': 0, 'Cochin': 1, 'Delhi': 2, 'Hyderabad': 3, 'Kolkata': 4, 'New Delhi': 5}


In [20]:
#stops
flight_train_df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', '4 stops'],
      dtype=object)

In [21]:
le_stops = preprocessing.LabelEncoder()
le_stops.fit(flight_train_df['Total_Stops'])
flight_train_df['Total_Stops'] = le_stops.transform(flight_train_df['Total_Stops'])
le_name_mapping = dict(zip(le_stops.classes_, le_stops.transform(le_stops.classes_)))
print(le_name_mapping)
#flight_train_df = flight_train_df.append(flight_train_df)

{'1 stop': 0, '2 stops': 1, '3 stops': 2, '4 stops': 3, 'non-stop': 4}


In [22]:
flight_train_df.head()

,Airline,Date_of_Journey,Source,Destination,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day_of_week
0,3,24/03/2019,0,5,22:20,01:10 22 Mar,2h 50m,4,No info,3897,7
1,1,1/05/2019,3,0,05:50,13:15,7h 25m,1,No info,7662,6
2,4,9/06/2019,2,1,09:25,04:25 10 Jun,19h,1,No info,13882,5
3,3,12/05/2019,3,0,18:05,23:30,5h 25m,0,No info,6218,4
4,3,01/03/2019,0,5,16:50,21:35,4h 45m,0,No info,13302,4


In [23]:
flight_train_df['Dep_Time'] = pd.to_datetime(flight_train_df['Dep_Time']).astype(int)
flight_train_df.head()

,Airline,Date_of_Journey,Source,Destination,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day_of_week
0,3,24/03/2019,0,5,1552947600000000000,01:10 22 Mar,2h 50m,4,No info,3897,7
1,1,1/05/2019,3,0,1552888200000000000,13:15,7h 25m,1,No info,7662,6
2,4,9/06/2019,2,1,1552901100000000000,04:25 10 Jun,19h,1,No info,13882,5
3,3,12/05/2019,3,0,1552932300000000000,23:30,5h 25m,0,No info,6218,4
4,3,01/03/2019,0,5,1552927800000000000,21:35,4h 45m,0,No info,13302,4


In [24]:
#flight_train_df['Duration'].unique()
flight_train_df['Duration'] = pd.to_timedelta(flight_train_df['Duration']).astype(int)
#flight_train_df.head()

In [25]:
flight_train_df['Additional_Info'].unique()

array(['No info', 'In-flight meal not included',
       'No check-in baggage included', '1 Short layover', 'No Info',
       '1 Long layover', 'Change airports', 'Business class',
       'Red-eye flight', '2 Long layover'], dtype=object)

In [26]:
Additional_Info = preprocessing.LabelEncoder()
Additional_Info.fit(flight_train_df['Additional_Info'])
flight_train_df['Additional_Info'] = Additional_Info.transform(flight_train_df['Additional_Info'])
le_name_mapping = dict(zip(Additional_Info.classes_, Additional_Info.transform(Additional_Info.classes_)))
print(le_name_mapping)

{'1 Long layover': 0, '1 Short layover': 1, '2 Long layover': 2, 'Business class': 3, 'Change airports': 4, 'In-flight meal not included': 5, 'No Info': 6, 'No check-in baggage included': 7, 'No info': 8, 'Red-eye flight': 9}


In [27]:
flight_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10682 entries, 0 to 10682
Data columns (total 11 columns):
Airline            10682 non-null int64
Date_of_Journey    10682 non-null object
Source             10682 non-null int64
Destination        10682 non-null int64
Dep_Time           10682 non-null int64
Arrival_Time       10682 non-null object
Duration           10682 non-null int64
Total_Stops        10682 non-null int64
Additional_Info    10682 non-null int64
Price              10682 non-null int64
day_of_week        10682 non-null int64
dtypes: int64(9), object(2)
memory usage: 1001.4+ KB


In [28]:
flight_train_df.head()

,Airline,Date_of_Journey,Source,Destination,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day_of_week
0,3,24/03/2019,0,5,1552947600000000000,01:10 22 Mar,10200000000000,4,8,3897,7
1,1,1/05/2019,3,0,1552888200000000000,13:15,26700000000000,1,8,7662,6
2,4,9/06/2019,2,1,1552901100000000000,04:25 10 Jun,68400000000000,1,8,13882,5
3,3,12/05/2019,3,0,1552932300000000000,23:30,19500000000000,0,8,6218,4
4,3,01/03/2019,0,5,1552927800000000000,21:35,17100000000000,0,8,13302,4


In [29]:
train_df = flight_train_df.drop(columns=['Date_of_Journey',  'Arrival_Time'])

In [30]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10682 entries, 0 to 10682
Data columns (total 9 columns):
Airline            10682 non-null int64
Source             10682 non-null int64
Destination        10682 non-null int64
Dep_Time           10682 non-null int64
Duration           10682 non-null int64
Total_Stops        10682 non-null int64
Additional_Info    10682 non-null int64
Price              10682 non-null int64
day_of_week        10682 non-null int64
dtypes: int64(9)
memory usage: 834.5 KB


### Load test data and apply the same transformation

In [31]:
#change working directory
#import os
#os.getcwd()
#os.chdir("/home/albin/data2/personal/my_interests/Flight_Ticket_Participant_Datasets")

#load the dataset
flight_test_df = pd.read_excel("Test_set.xlsx", na_values= '?')

#view top 10 rows
flight_test_df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info
0,Jet Airways,6/06/2019,Delhi,Cochin,DEL → BOM → COK,17:30,04:25 07 Jun,10h 55m,1 stop,No info
1,IndiGo,12/05/2019,Kolkata,Banglore,CCU → MAA → BLR,06:20,10:20,4h,1 stop,No info
2,Jet Airways,21/05/2019,Delhi,Cochin,DEL → BOM → COK,19:15,19:00 22 May,23h 45m,1 stop,In-flight meal not included
3,Multiple carriers,21/05/2019,Delhi,Cochin,DEL → BOM → COK,08:00,21:00,13h,1 stop,No info
4,Air Asia,24/06/2019,Banglore,Delhi,BLR → DEL,23:55,02:45 25 Jun,2h 50m,non-stop,No info


In [32]:
flight_test_df.drop(columns='Route', inplace=True)

In [33]:
flight_test_df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        0
Additional_Info    0
dtype: int64

In [34]:
flight_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2671 entries, 0 to 2670
Data columns (total 9 columns):
Airline            2671 non-null object
Date_of_Journey    2671 non-null object
Source             2671 non-null object
Destination        2671 non-null object
Dep_Time           2671 non-null object
Arrival_Time       2671 non-null object
Duration           2671 non-null object
Total_Stops        2671 non-null object
Additional_Info    2671 non-null object
dtypes: object(9)
memory usage: 187.9+ KB


In [35]:
flight_test_df['day_of_week'] = pd.to_datetime(flight_test_df['Date_of_Journey']).dt.day_name()
days = {'Monday':1, 'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7}
flight_test_df['day_of_week'] = flight_test_df['day_of_week'].apply(lambda x: days[x])
flight_test_df.head()

,Airline,Date_of_Journey,Source,Destination,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,day_of_week
0,Jet Airways,6/06/2019,Delhi,Cochin,17:30,04:25 07 Jun,10h 55m,1 stop,No info,4
1,IndiGo,12/05/2019,Kolkata,Banglore,06:20,10:20,4h,1 stop,No info,4
2,Jet Airways,21/05/2019,Delhi,Cochin,19:15,19:00 22 May,23h 45m,1 stop,In-flight meal not included,2
3,Multiple carriers,21/05/2019,Delhi,Cochin,08:00,21:00,13h,1 stop,No info,2
4,Air Asia,24/06/2019,Banglore,Delhi,23:55,02:45 25 Jun,2h 50m,non-stop,No info,1


In [36]:
flight_test_df['Airline'] = le_Airline.transform(flight_test_df['Airline'])
le_name_mapping = dict(zip(le_Airline.classes_, le_Airline.transform(le_Airline.classes_)))
print(le_name_mapping)

{'Air Asia': 0, 'Air India': 1, 'GoAir': 2, 'IndiGo': 3, 'Jet Airways': 4, 'Jet Airways Business': 5, 'Multiple carriers': 6, 'Multiple carriers Premium economy': 7, 'SpiceJet': 8, 'Trujet': 9, 'Vistara': 10, 'Vistara Premium economy': 11}


In [37]:
flight_test_df['Source'] = le_Source.transform(flight_test_df['Source'])
le_name_mapping = dict(zip(le_Source.classes_, le_Source.transform(le_Source.classes_)))
print(le_name_mapping)

{'Banglore': 0, 'Chennai': 1, 'Delhi': 2, 'Kolkata': 3, 'Mumbai': 4}


In [38]:
flight_test_df['Destination'] = le_Destination.transform(flight_test_df['Destination'])
le_name_mapping = dict(zip(le_Destination.classes_, le_Destination.transform(le_Destination.classes_)))
print(le_name_mapping)

{'Banglore': 0, 'Cochin': 1, 'Delhi': 2, 'Hyderabad': 3, 'Kolkata': 4, 'New Delhi': 5}


In [39]:
flight_test_df['Dep_Time'] = pd.to_datetime(flight_test_df['Dep_Time']).astype(int)

In [40]:
flight_test_df['Duration'] = pd.to_timedelta(flight_test_df['Duration']).astype(int)

In [41]:
le_stops = preprocessing.LabelEncoder()
le_stops.fit(flight_test_df['Total_Stops'])
flight_test_df['Total_Stops'] = le_stops.transform(flight_test_df['Total_Stops'])
le_name_mapping = dict(zip(le_stops.classes_, le_stops.transform(le_stops.classes_)))
print(le_name_mapping)

{'1 stop': 0, '2 stops': 1, '3 stops': 2, '4 stops': 3, 'non-stop': 4}


In [42]:
Additional_Info = preprocessing.LabelEncoder()
Additional_Info.fit(flight_test_df['Additional_Info'])
flight_test_df['Additional_Info'] = Additional_Info.transform(flight_test_df['Additional_Info'])
le_name_mapping = dict(zip(Additional_Info.classes_, Additional_Info.transform(Additional_Info.classes_)))
print(le_name_mapping)

{'1 Long layover': 0, 'Business class': 1, 'Change airports': 2, 'In-flight meal not included': 3, 'No check-in baggage included': 4, 'No info': 5}


In [43]:
test_df = flight_test_df.drop(columns=['Date_of_Journey', 'Arrival_Time'])

In [44]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2671 entries, 0 to 2670
Data columns (total 8 columns):
Airline            2671 non-null int64
Source             2671 non-null int64
Destination        2671 non-null int64
Dep_Time           2671 non-null int64
Duration           2671 non-null int64
Total_Stops        2671 non-null int64
Additional_Info    2671 non-null int64
day_of_week        2671 non-null int64
dtypes: int64(8)
memory usage: 167.0 KB


In [45]:
X_train = train_df.drop(columns='Price')
y_train = train_df['Price']

In [46]:
#Sample test

X_sam_train, X_sam_test, y_sam_train, y_sam_test = train_test_split(X_train, y_train, test_size=.3, random_state=42)

In [47]:
from sklearn.ensemble import RandomForestRegressor
#Make ML model the instance
model = RandomForestRegressor(random_state=7)
#Hyper Parameters Set
params = {'n_estimators':[10,15,20,25,30],
          'max_depth': [5, 15, 25, 50], 
          'min_samples_leaf':[1,2,3],
          'min_samples_split':[3,4,5,6,7], 
          #'max_features':[5, 10, 20]
         }
#Make ML model with hyper parameters sets
model1 = GridSearchCV(model, param_grid=params, n_jobs=-1)
#Train the model
model1.fit(X_sam_train, y_sam_train)
#The best hyper parameters set
print("Best Hyper Parameters:",model1.best_params_)

#The best estimator
print("Best estimator:", model1.best_estimator_)

#Prediction
prediction=model1.predict(X_sam_test)
## Evaluation
# Accuracy


/usr/lib/python3/dist-packages/sklearn/ensemble/weight_boosting.py:29: DeprecationWarning: numpy.core.umath_tests is an internal NumPy module and should not be imported. It will be removed in a future NumPy release.
  from numpy.core.umath_tests import inner1d


Best Hyper Parameters: {'max_depth': 50, 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 30}
Best estimator: RandomForestRegressor(bootstrap=True, criterion='mse', max_depth=50,
           max_features='auto', max_leaf_nodes=None,
           min_impurity_decrease=0.0, min_impurity_split=None,
           min_samples_leaf=1, min_samples_split=6,
           min_weight_fraction_leaf=0.0, n_estimators=30, n_jobs=1,
           oob_score=False, random_state=7, verbose=0, warm_start=False)


In [48]:
print("Accuracy:", model1.score(X_sam_test, y_sam_test))
#5 0.9434138669690630
#4 0.9407584697743018
#3 0.9409155520986593

Accuracy: 0.761022726979905


In [49]:
#Prediction
model1.fit(X_train, y_train)
prediction=model1.predict(test_df)
pd.DataFrame(prediction, columns = ['Price']).to_excel("Final_Pred.xlsx", index = False)
print(pd.DataFrame(prediction).head())

              0
0   9608.986498
1   4561.599735
2  27372.707446
3  11468.564897
4   3992.890603
